# Step 1 — Disfluency BIO labeling (IP/RP/RM/RC/FS taxonomy)

`data/normalized/intent_dataset_normalized.jsonl` already carries a gold `disfluencies` field per record — each entry is `{"tag": <IP|FS|RC|RP|RM>, "token"|"tokens": <span text>}` describing exactly which surface span carries which disfluency type. This notebook converts that gold annotation directly into token-level BIO tags over `text` (a prior version of this notebook re-derived labels via regex heuristics, which silently missed most `RP`/`RM` repairs and any non-`eh`/`anu` bridge filler — using the gold field instead is both more accurate and simpler).

Taxonomy:
- `IP` (interregnum/filler) — discourse fillers, single or multi-word (`eh`, `anu`, `um`, `hmm`, `ee`, `em`, `mmm`, `eh ya`)
- `FS` (false start) — hyphen-truncated partial word immediately followed by its completed form (e.g. `mi- mie`)
- `RC` (repeat) — exact adjacent word duplication (e.g. `pesan, pesan`)
- `RP`/`RM` (reparandum/repair) — self-correction pairs: `RP` is the wrong value, `RM` is its correction

Encoder target for downstream training: `indobenchmark/indobert-base-p1` (matches the NER backbone, reverted in `9f1dd1a`).

In [1]:
import json
from collections import Counter
from pathlib import Path

SRC = Path("../data/normalized/intent_dataset_normalized.jsonl")
OUT_DIR = Path("../data/disfluency")

## Step 2 — Span matching helpers

Each gold `disfluencies` entry holds the disfluency span as raw text (`token` for single/multi-word spans, `tokens` for `RC` repeat pairs). `strip_punct` normalizes both the sentence tokens and the gold span text the same way before matching, so trailing commas on tokens like `"eh,"` don't break equality.

In [2]:
def tokenize(text: str) -> list[str]:
    return text.split()


def strip_punct(tok: str) -> str:
    return tok.strip(",.;:!?").lower()


def get_span_tokens(d: dict) -> list[str]:
    if "tokens" in d:
        return [strip_punct(t) for t in d["tokens"]]
    return [strip_punct(t) for t in d["token"].split()]

## Step 3 — BIO labeler from gold spans

`match_spans` locates each gold disfluency's token span inside the sentence and marks those positions used, so a repeated surface form (e.g. `eh` appearing twice, once inside `eh ya` and once standalone) binds to the correct occurrence instead of always the first. Gold entries aren't guaranteed to be in left-to-right text order (verified: 1/5100 records has `RP`/`RM`/`IP` listed out of text order), so matching is order-independent over the disfluencies list rather than a single sequential scan.

In [3]:
def match_spans(toks: list[str], disfluencies: list[dict]) -> list[tuple[int, int, str]] | None:
    """Match each gold disfluency entry to a token span, returning (start, end, tag)
    triples (end exclusive). Returns None if any entry can't be matched."""
    used = [False] * len(toks)
    spans = []
    for d in disfluencies:
        span = get_span_tokens(d)
        L = len(span)
        found = -1
        for i in range(len(toks) - L + 1):
            if any(used[i : i + L]):
                continue
            if toks[i : i + L] == span:
                found = i
                break
        if found == -1:
            return None
        for k in range(found, found + L):
            used[k] = True
        spans.append((found, found + L, d["tag"]))
    return spans


def full_label(raw_text: str, disfluencies: list[dict]) -> tuple[list[str], list[str]]:
    raw_toks = tokenize(raw_text)
    toks = [strip_punct(t) for t in raw_toks]
    labels = ["O"] * len(toks)
    spans = match_spans(toks, disfluencies)
    if spans is None:
        return raw_toks, None
    for start, end, tag in spans:
        labels[start] = f"B-{tag}"
        for k in range(start + 1, end):
            labels[k] = f"I-{tag}"
    return raw_toks, labels

## Step 4 — Validation

Auto-corrects orphan `I-` tags (continuation tag with no preceding `B-`/`I-` of the same type) to `B-`, and any unrecognized label value to `O`. `match_spans`/`full_label` never produce orphans by construction (each matched span always starts with `B-`), but this guards against future changes silently breaking BIO well-formedness.

In [4]:
VALID_LABELS = {"O", "B-IP", "I-IP", "B-FS", "B-RC", "I-RC", "B-RP", "B-RM"}


def validate_bio(toks: list[str], labels: list[str], record_id) -> list[str]:
    fixed = list(labels)
    prev_type = None
    for i, lb in enumerate(fixed):
        if lb not in VALID_LABELS:
            print(f"[{record_id}] unknown label {lb!r} at token {toks[i]!r}, coercing to O")
            fixed[i] = "O"
            lb = "O"
        if lb.startswith("I-"):
            tag_type = lb[2:]
            if prev_type != tag_type:
                print(f"[{record_id}] orphan {lb} at token {toks[i]!r}, coercing to B-{tag_type}")
                fixed[i] = f"B-{tag_type}"
        prev_type = fixed[i][2:] if fixed[i] != "O" else None
    return fixed

## Step 5 — Run labeler over the corpus

In [5]:
rows = [json.loads(line) for line in SRC.open()]

labeled_rows = []
tag_counts = Counter()
unmatched = []
for row in rows:
    toks, labels = full_label(row["text"], row.get("disfluencies", []))
    if labels is None:
        unmatched.append(row["id"])
        continue
    labels = validate_bio(toks, labels, row["id"])
    for lb in labels:
        tag_counts[lb] += 1
    labeled_rows.append({
        "id": row["id"],
        "text": row["text"],
        "intent": row["intent"],
        "tokens": toks,
        "labels": labels,
    })

print(f"{len(labeled_rows)} records labeled, {len(unmatched)} unmatched: {unmatched}")
print("tag counts:", dict(tag_counts))

5100 records labeled, 0 unmatched: []
tag counts: {'B-IP': 4071, 'O': 32614, 'B-RC': 1245, 'I-RC': 1245, 'B-FS': 1270, 'B-RP': 29, 'B-RM': 29, 'I-IP': 508}


## Step 6 — Sanity check on sample records

Spot-checks the gold-to-BIO mapping against representative records for each tag, including a multi-word `eh ya` `IP` span and a genuine `RP`/`RM` repair pair.

In [6]:
sample_ids = [1, 3, 2677, 7]
samples_by_id = {row["id"]: row for row in rows}

for rid in sample_ids:
    row = samples_by_id[rid]
    toks, labels = full_label(row["text"], row.get("disfluencies", []))
    print(row["text"])
    print([f"{t}/{l}" for t, l in zip(toks, labels) if l != "O"] or "no tags")
    print()

eh, saya mau pesan nasi goreng spesial dua porsi sama ayam bakar satu
['eh,/B-IP']

pesan pesan nasi uduk satu sama tahu tempe goreng dua ya kak
['pesan/B-RC', 'pesan/I-RC']

eh ya, ada pempek versi pedas eh, gak?
['eh/B-IP', 'ya,/I-IP', 'eh,/B-IP']

pesan satu, eh, dua porsi nasi goreng seafood sama satu jus alpukat
['satu,/B-RP', 'eh,/B-IP', 'dua/B-RM']



## Step 7 — Align labels to IndoBERT WordPiece tokens

`indobenchmark/indobert-base-p1` — plain IndoBERT, matches the NER backbone (reverted in `9f1dd1a`). Word-level BIO labels align to WordPiece subwords via `word_ids()`: the first subword of each word keeps its label id, continuation subwords and special tokens get `-100` so `CrossEntropyLoss` ignores them during training.

In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label_list = sorted({lb for r in labeled_rows for lb in r["labels"]})
label2id = {lb: i for i, lb in enumerate(label_list)}
id2label = {i: lb for lb, i in label2id.items()}
print(label2id)

{'B-FS': 0, 'B-IP': 1, 'B-RC': 2, 'B-RM': 3, 'B-RP': 4, 'I-IP': 5, 'I-RC': 6, 'O': 7}


In [8]:
def align_labels_with_tokens(tokens: list[str], labels: list[str]) -> dict:
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    word_ids = encoding.word_ids()
    aligned = []
    prev_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned.append(-100)
        elif word_id != prev_word_id:
            aligned.append(label2id[labels[word_id]])
        else:
            aligned.append(-100)
        prev_word_id = word_id
    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": aligned,
    }


aligned_rows = []
mismatches = 0
for r in labeled_rows:
    aligned = align_labels_with_tokens(r["tokens"], r["labels"])
    if len(aligned["input_ids"]) != len(aligned["labels"]):
        mismatches += 1
        continue
    aligned_rows.append({
        "id": r["id"],
        "intent": r["intent"],
        **aligned,
    })

print(f"{len(aligned_rows)} aligned, {mismatches} mismatches")

5100 aligned, 0 mismatches


## Step 8 — Dominant disfluency type per record (for stratified split)

Each record gets one dominant tag for stratification — the rarest non-`O` tag type present, falling back to `fluent` for records with no disfluency tags. Prioritizing the rarest tag (rather than e.g. first-occurring) keeps `FS`/`RP`/`RM` (only 3/2/2 occurrences each) from being randomly orphaned into a single split.

In [9]:
TAG_PRIORITY = ["FS", "RP", "RM", "RC", "IP"]


def dominant_type(labels: list[str]) -> str:
    present = {lb[2:] for lb in labels if lb != "O"}
    for tag in TAG_PRIORITY:
        if tag in present:
            return tag
    return "fluent"


dominant_by_id = {r["id"]: dominant_type(r["labels"]) for r in labeled_rows}
print(Counter(dominant_by_id.values()))

Counter({'IP': 2556, 'FS': 1270, 'RC': 1245, 'RP': 29})


## Step 9 — Stratified 80/10/10 split

`FS`, `RP`, `RM` have only 2-3 records each. `train_test_split` needs >=2 members per class per split, and this runs as *two* sequential splits (80/20, then the 20% into 10/10) — a class needs >=4 total to survive both. Records whose dominant type has fewer than 4 occurrences go entirely to train; the remaining (`fluent`, `IP`, `RC`) stratify normally.

In [10]:
from sklearn.model_selection import train_test_split

SEED = 42
VAL_FRAC = 0.1
TEST_FRAC = 0.1
MIN_STRATIFY_COUNT = 4

counts = Counter(dominant_by_id.values())
forced_train_ids = {rid for rid, dt in dominant_by_id.items() if counts[dt] < MIN_STRATIFY_COUNT}
stratifiable = [r for r in aligned_rows if r["id"] not in forced_train_ids]
forced_train = [r for r in aligned_rows if r["id"] in forced_train_ids]

labels_strat = [dominant_by_id[r["id"]] for r in stratifiable]
train, rest = train_test_split(
    stratifiable, test_size=VAL_FRAC + TEST_FRAC, stratify=labels_strat, random_state=SEED
)
rest_labels = [dominant_by_id[r["id"]] for r in rest]
val, test = train_test_split(
    rest, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=rest_labels, random_state=SEED
)
train = train + forced_train

print(f"train={len(train)} val={len(val)} test={len(test)}")

train=4080 val=510 test=510


In [11]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


OUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(OUT_DIR / "train.jsonl", train)
write_jsonl(OUT_DIR / "val.jsonl", val)
write_jsonl(OUT_DIR / "test.jsonl", test)
json.dump(label2id, (OUT_DIR / "label_map.json").open("w"), indent=2)

print("wrote", OUT_DIR)

wrote ../data/disfluency
